In [10]:
#Multi-Head Attention
import torch
import math
from torch import nn
from d2l import torch as d2l

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self,key_size,query_size,value_size,num_heads,num_hiddens,droupout,bias=False,**kwargs):
        super(MultiHeadAttention,self).__init__()
        self.num_heads=num_heads
        self.attention = d2l.DotProductAttention(droupout)
        self.W_q=nn.Linear(query_size,num_hiddens,bias=bias)
        self.W_k=nn.Linear(key_size,num_hiddens,bias=bias)
        self.W_v=nn.Linear(value_size,num_hiddens,bias=bias)
        self.W_o=nn.Linear(num_hiddens,num_hiddens,bias=bias)

    def forward(self,queries,keys,values,valid_lens):
        queries=transport_qkv(self.W_q(queries),self.num_heads)
        keys=transport_qkv(self.W_k(keys),self.num_heads)
        values=transport_qkv(self.W_v(values),self.num_heads)

        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(valid_lens,repeats=self.num_heads,dim=0)
        output = self.attention(queries,keys,values,valid_lens)
        out = transport_out(output,self.num_heads)
        return self.W_o(out)

In [18]:
def transport_qkv(X,num_heads):
    X=X.reshape(X.shape[0],X.shape[1],num_heads,-1)
    X=X.permute(0,2,1,3)
    return X.reshape(-1,X.shape[2],X.shape[3])

def transport_out(X,num_heads):
    X=X.reshape(-1,num_heads,X.shape[1],X.shape[2])
    X=X.permute(0,2,1,3)
    return X.reshape(X.shape[0],X.shape[1],-1)

In [19]:
num_heads,num_hiddens=5,100
attention = MultiHeadAttention(num_hiddens,num_hiddens,num_hiddens,num_heads,num_hiddens,0.5)
batch_size,num_q,num_kv,valid_lens=2,4,6,torch.tensor([2,3])
X=torch.ones(batch_size,num_q,num_hiddens)
Y=torch.ones(batch_size,num_kv,num_hiddens)
out=attention(X,Y,Y,valid_lens)
out.shape

torch.Size([2, 4, 100])